# Tuần 1 — Mở khoá Tầng 2  ·  bản 2

**Sửa so với bản 1** (phát hiện khi chạy thật ngày 12/11):

| | |
|---|---|
| `DATA` | `.../Sym/data/DISFA` → **`.../Sym/DISFA`** (bản 1 thừa một cấp `data/`) |
| Ô thiết lập | kiểm cả `list/`, `img/`, `splits/*.json` chứ không chỉ thư mục gốc; `assert` dừng ngay nếu sai |
| Ô dò (bước 2) | thông báo lỗi cũ đổ oan cho tên tham số; nay phân loại theo traceback |
| Ô chẩn đoán help/hurt | bản 1 phán **NO-GO khi chưa có dữ liệu** — nay chặn lại |
| Kiểm toàn vẹn | đối chiếu subject trong npz với `val_subjects` / `test_subjects` của manifest |
| Subject | suy được từ `paths` (`p.split('/')[0]`) nếu npz thiếu khoá `subjects` |

**Đã xác nhận:** `--split {val,test}` là tham số hợp lệ.

Các ô đều idempotent — chạy lại chỉ bỏ qua phần đã xong.

## 0. Thiết lập

Ô này **dừng ngay** nếu có đường dẫn sai, thay vì để bạn chạy 9 lệnh rồi mới thấy cả 9 cùng hỏng.

In [ ]:
import os, sys, json, subprocess, time, itertools
from pathlib import Path
from collections import defaultdict
import numpy as np

try:
    from google.colab import drive
    if not Path('/content/drive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass

# ================== ĐƯỜNG DẪN ==================
SYM    = Path('/content/drive/MyDrive/Sym')
REPO   = SYM / 'Contrastive_V1'        # nơi có train_contrastive.py
RUNS   = SYM / 'runs_retry1'
DATA   = SYM / 'DISFA'                 # <-- ĐÃ SỬA: không có cấp 'data/'
SPLITS = SYM / 'splits'
BATCH  = 64

VARIANTS = ['baseline', 'current', 'full']
FOLDS    = [1, 2, 3]
SEEDS    = [0, 1, 2]
AU_NAMES = ['AU1', 'AU2', 'AU4', 'AU6', 'AU9', 'AU12', 'AU25', 'AU26']
# ===============================================

TRAIN_PY = REPO / 'train_contrastive.py'

checks = [('train_contrastive.py', TRAIN_PY,            TRAIN_PY.is_file()),
          ('RUNS',                 RUNS,                RUNS.is_dir()),
          ('DATA',                 DATA,                DATA.is_dir()),
          ('DATA/list/',           DATA/'list',         (DATA/'list').is_dir()),
          ('DATA/img/',            DATA/'img',          (DATA/'img').is_dir()),
          ('SPLITS',               SPLITS,              SPLITS.is_dir())]
checks += [(f'splits/fold{f}.json', SPLITS/f'fold{f}.json',
            (SPLITS/f'fold{f}.json').is_file()) for f in FOLDS]

bad = 0
for name, p, ok in checks:
    print(f'{"OK   " if ok else "THIẾU"}  {name:22s} {p}')
    bad += (not ok)

def run_logged(cmd, cwd=None):
    """Chạy lệnh, in log, raise nếu lỗi."""
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    t0 = time.time()
    p = subprocess.run([str(c) for c in cmd], cwd=str(cwd or REPO),
                       capture_output=True, text=True)
    if p.stdout: print(p.stdout[-4000:])
    if p.returncode != 0:
        print('--- STDERR ---'); print(p.stderr[-4000:])
        raise RuntimeError(f'mã lỗi {p.returncode}')
    print(f'[xong sau {time.time()-t0:.1f}s]\n', flush=True)
    return p

def subjects_of(d):
    """Subject của từng frame. Ưu tiên khoá 'subjects', nếu không thì lấy
    thành phần đầu của đường dẫn ảnh — đúng cách experiment_data.py làm."""
    if 'subjects' in d:
        return np.asarray(d['subjects']).astype(str)
    if 'paths' in d:
        return np.array([str(p).split('/')[0] for p in d['paths']])
    return None

def load_npz(p):
    d = np.load(p, allow_pickle=True)
    return {k: d[k] for k in d.files}

assert bad == 0, f'{bad} đường dẫn sai — sửa rồi chạy lại ô này (ô chẩn đoán bên dưới sẽ giúp tìm)'
print('\n>>> Đường dẫn hợp lệ.')

## 0b. Chẩn đoán đường dẫn — chỉ chạy nếu ô trên báo THIẾU

In [ ]:
print('=== Cây thư mục dưới SYM ===')
for p in sorted(SYM.iterdir()):
    print('   ', p.name + ('/' if p.is_dir() else ''))

print('\n=== Tìm file list ở mọi nơi ===')
hits = sorted(SYM.rglob('DISFA_train_img_path_fold*.txt'))
for h in hits[:10]: print('   ', h)
print(f'\n>>> DATA đúng phải là: {hits[0].parent.parent}' if hits
      else '>>> KHÔNG tìm thấy list/ — cần sinh lại bằng tool/DISFA_image_label_process.py')

print('\n=== Tìm train_contrastive.py ===')
for h in sorted(SYM.rglob('train_contrastive.py'))[:5]: print('   ', h)

print('\n=== Tìm manifest ===')
for h in sorted(SYM.rglob('fold*.json'))[:10]: print('   ', h)

## 1. Tình trạng hiện tại

- `--` chưa train  ·  `..` có ckpt chưa evaluate  ·  `T.` có test thiếu val  ·  `TV` đủ cả hai

In [ ]:
def find_npz(d, kind):
    if not d.exists(): return None
    for pat in (f'{kind}_predictions.npz', f'*{kind}*.npz'):
        hits = sorted(d.glob(pat))
        if hits: return hits[0]
    return None

def run_state(variant, fold, seed):
    out = RUNS / variant / f'fold{fold}' / f'seed{seed}'
    return dict(out=out, ckpt=out/'best.pt',
                test=find_npz(out/'evaluation', 'test'),
                val =find_npz(out/'evaluation_val', 'val') or find_npz(out/'evaluation', 'val'))

rows, need_val, need_train = [], [], []
for v in VARIANTS:
    for f in FOLDS:
        for s in SEEDS:
            st = run_state(v, f, s)
            has = (st['ckpt'].exists(), st['test'] is not None, st['val'] is not None)
            rows.append(has)
            if   not has[0]: need_train.append((v, f, s))
            elif not has[2]: need_val.append((v, f, s))

w = max(len(v) for v in VARIANTS) + 2
hdr = f'{"variant":<{w}}{"fold":>5}   ' + '  '.join(f'{"s"+str(s):>4}' for s in SEEDS)
print(hdr); print('-'*len(hdr))
for v in VARIANTS:
    for f in FOLDS:
        cs = []
        for s in SEEDS:
            st = run_state(v, f, s)
            cs.append(f'{"TV":>4}' if st['val'] is not None else
                      f'{"T.":>4}' if st['test'] is not None else
                      f'{"..":>4}' if st['ckpt'].exists() else f'{"--":>4}')
        print(f'{v:<{w}}{f:>5}   ' + '  '.join(cs))

n = len(rows)
print(f'\nCó best.pt : {sum(r[0] for r in rows)}/{n}')
print(f'Có test    : {sum(r[1] for r in rows)}/{n}')
print(f'Có val     : {sum(r[2] for r in rows)}/{n}   <-- cần bằng số "có best.pt"')
print(f'\nCần sinh val : {len(need_val)} run')
print(f'Chưa train   : {len(need_train)} run')

## 2. Dò lệnh trên MỘT run

Ghi ra thư mục **riêng** `evaluation_val/` để không đụng `test_metrics.json` — file đó là mốc kiểm `C0`.

In [ ]:
if not need_val:
    print('Không run nào cần sinh val. Bỏ qua bước 2 và 3.')
else:
    pv, pf, ps = need_val[0]
    print(f'Dò trên: {pv} / fold{pf} / seed{ps}\n')
    st  = run_state(pv, pf, ps)
    out = st['out']; ev_val = out/'evaluation_val'
    before = sorted(p.name for p in (out/'evaluation').glob('*')) if (out/'evaluation').exists() else []

    try:
        run_logged([sys.executable, TRAIN_PY, 'evaluate',
                    '--split', 'val',
                    '--checkpoint', out/'best.pt',
                    '--data-root',  DATA,
                    '--manifest',   SPLITS/f'fold{pf}.json',
                    '--output',     ev_val,
                    '--batch-size', BATCH])
    except RuntimeError as e:
        print('!! Thất bại. Đối chiếu traceback phía trên:')
        print('   FileNotFoundError .../list/...  -> sai --data-root  -> chạy ô 0b')
        print('   ValueError: Data changed ...    -> file list KHÁC lúc tạo manifest')
        print('                                      ĐỪNG sinh lại manifest, báo lại trước')
        print('   unrecognized arguments          -> sai tên tham số  -> chạy ô --help')
        print('   CUDA out of memory              -> giảm BATCH xuống 32\n', e)

    after = sorted(p.name for p in (out/'evaluation').glob('*')) if (out/'evaluation').exists() else []
    print('evaluation_val/  :', sorted(p.name for p in ev_val.glob('*')) if ev_val.exists() else 'KHÔNG TẠO ĐƯỢC')
    print('evaluation/ trước:', before)
    print('evaluation/ sau  :', after)
    print('\n>>> evaluation/ KHÔNG đổi — an toàn' if before == after
          else '\n>>> CẢNH BÁO: evaluation/ đã bị thay đổi, DỪNG LẠI')

In [ ]:
# Chỉ chạy nếu nghi sai tên tham số
run_logged([sys.executable, TRAIN_PY, 'evaluate', '--help'])

## 3. Sinh `val_predictions.npz` cho tất cả run

Suy luận thuần, không train. Val chỉ 3 subject nên nhanh hơn test khoảng ba lần.

In [ ]:
t0, done, failed = time.time(), [], []
for (v, f, s) in list(need_val):
    st = run_state(v, f, s)
    if st['val'] is not None:
        continue
    try:
        run_logged([sys.executable, TRAIN_PY, 'evaluate',
                    '--split', 'val',
                    '--checkpoint', st['out']/'best.pt',
                    '--data-root',  DATA,
                    '--manifest',   SPLITS/f'fold{f}.json',
                    '--output',     st['out']/'evaluation_val',
                    '--batch-size', BATCH])
        done.append((v, f, s))
    except RuntimeError:
        failed.append((v, f, s))
        if len(failed) == 1:
            print('>>> Run đầu tiên đã lỗi. Dừng để khỏi lặp lại 9 lần.')
            break

print(f'\nXong {len(done)} run trong {(time.time()-t0)/60:.1f} phút')
if failed: print(f'THẤT BẠI: {failed}')

## 4. Kiểm toàn vẹn

`load_manifest` đã tự cưỡng chế phần lớn: hash từng file `list/`, hash `M_AE`, `fit`/`val` là phân hoạch rời rạc, không chồng subject giữa `fit`/`val`/`test`. Ô này kiểm những thứ **còn lại**, nằm ngoài tầm với của nó.

In [ ]:
problems = []
for v in VARIANTS:
    for f in FOLDS:
        for s in SEEDS:
            st = run_state(v, f, s)
            if st['val'] is None or st['test'] is None: continue
            tag = f'{v}/fold{f}/seed{s}'
            dv, dt = load_npz(st['val']), load_npz(st['test'])

            for need in ('labels', 'probabilities'):
                for nm, d in (('val', dv), ('test', dt)):
                    if need not in d:
                        problems.append(f'{tag}: {nm} thiếu khoá "{need}" (có: {list(d)})')
            if 'labels' not in dv or 'probabilities' not in dv: continue

            if dv['labels'].shape != dv['probabilities'].shape:
                problems.append(f'{tag}: val labels{dv["labels"].shape} != probs{dv["probabilities"].shape}')

            pr = dv['probabilities']
            if pr.min() < -1e-6 or pr.max() > 1+1e-6:
                problems.append(f'{tag}: val probs ngoài [0,1]: [{pr.min():.3f}, {pr.max():.3f}]')
            if pr.min() >= 0.49:
                problems.append(f'{tag}: val probs >= 0.49 — DẤU HIỆU SIGMOID HAI LẦN')

            # đối chiếu với manifest: đúng split, đúng số frame
            mf = json.loads((SPLITS/f'fold{f}.json').read_text(encoding='utf-8'))
            sv, stt = subjects_of(dv), subjects_of(dt)
            if sv is not None:
                exp, got = set(map(str, mf['val_subjects'])), set(sv)
                if exp != got:
                    problems.append(f'{tag}: subject val {sorted(got)} != manifest {sorted(exp)}')
            if stt is not None:
                exp, got = set(map(str, mf['test_subjects'])), set(stt)
                if exp != got:
                    problems.append(f'{tag}: subject test {sorted(got)} != manifest {sorted(exp)}')
            if len(dv['labels']) != len(mf['val_indices']):
                problems.append(f'{tag}: val có {len(dv["labels"])} frame, manifest ghi {len(mf["val_indices"])}')
            if sv is not None and stt is not None and (set(sv) & set(stt)):
                problems.append(f'{tag}: val và test CHUNG subject {sorted(set(sv)&set(stt))} — rò rỉ!')

            ck = st['ckpt'].stat().st_mtime
            for nm, p in (('val', st['val']), ('test', st['test'])):
                if p.stat().st_mtime < ck:
                    problems.append(f'{tag}: {nm}.npz CŨ HƠN best.pt — sinh từ checkpoint khác')

ex = next((run_state(v,f,s)['val'] for v in VARIANTS for f in FOLDS for s in SEEDS
           if run_state(v,f,s)['val'] is not None), None)
if ex is not None:
    d = load_npz(ex)
    print(f'--- cấu trúc {ex.name} ---')
    for k, a in d.items():
        print(f'  {k:16s} shape={getattr(a,"shape","?")}  dtype={getattr(a,"dtype","?")}')
    sv = subjects_of(d)
    if sv is not None: print(f'  subject trong file: {sorted(set(map(str, sv)))}')
else:
    print('Chưa có file val nào — chạy bước 3 trước.')

print('\n' + ('>>> KHÔNG CÓ VẤN ĐỀ' if not problems else f'>>> {len(problems)} VẤN ĐỀ:'))
for p in problems: print('   -', p)

## 5. Chẩn đoán `help` / `hurt` — GO/NO-GO của Tầng 2

Với mỗi cặp AU, đếm trên inner val:

- **`help`** — detector vi phạm luật, nhãn thật thoả → luật **sửa được**
- **`hurt`** — detector vi phạm, nhãn thật cũng vi phạm → ngoại lệ thật, luật sẽ **phá** dự đoán đúng

Duyệt **toàn bộ** 28 cặp loại trừ và 56 kéo theo, không lọc trước theo lift — câu hỏi lúc này là "có chỗ nào cho luật sửa không", chưa phải "luật nào đạt chuẩn thống kê".

Con số quyết định là **số frame phân biệt**, không phải tổng qua các cặp: một frame bật nhầm ba AU dính tới ba cặp, cộng lại là đếm trùng.

In [ ]:
TH = 0.5
agg_excl = defaultdict(lambda: dict(help=0, hurt=0, viol=0))
agg_impl = defaultdict(lambda: dict(help=0, hurt=0, viol=0))
per_subject_help = defaultdict(int)
n_frames = n_runs = fr_help = fr_hurt = 0

for v in VARIANTS:
    for f in FOLDS:
        for s in SEEDS:
            st = run_state(v, f, s)
            if st['val'] is None: continue
            d  = load_npz(st['val'])
            y  = (d['labels'] > 0.5)
            yh = (d['probabilities'] > TH)
            subj = subjects_of(d)
            if subj is None: subj = np.zeros(len(y), dtype=int).astype(str)
            n_frames += len(y); n_runs += 1
            any_h = np.zeros(len(y), bool); any_k = np.zeros(len(y), bool)

            for i, j in itertools.combinations(range(len(AU_NAMES)), 2):
                viol = yh[:, i] & yh[:, j]
                both = y[:, i] & y[:, j]
                h = viol & ~both
                k = (AU_NAMES[i], AU_NAMES[j])
                agg_excl[k]['viol'] += int(viol.sum())
                agg_excl[k]['help'] += int(h.sum())
                agg_excl[k]['hurt'] += int((viol & both).sum())
                any_h |= h; any_k |= (viol & both)
                for sb in np.unique(subj[h]):
                    per_subject_help[str(sb)] += int((subj[h] == sb).sum())

            for i, j in itertools.permutations(range(len(AU_NAMES)), 2):
                viol = yh[:, i] & ~yh[:, j]
                tv   = y[:, i] & ~y[:, j]
                k = (AU_NAMES[i], AU_NAMES[j])
                agg_impl[k]['viol'] += int(viol.sum())
                agg_impl[k]['help'] += int((viol & ~tv).sum())
                agg_impl[k]['hurt'] += int((viol & tv).sum())

            fr_help += int(any_h.sum()); fr_hurt += int(any_k.sum())

if n_runs == 0:
    print('>>> CHƯA CÓ val_predictions.npz nào.')
    print('    Chạy xong bước 3 rồi quay lại ô này. KHÔNG có kết luận nào ở đây.')
else:
    def report(agg, title, arrow, topn=12):
        print(f'\n=== {title} ===')
        rank = sorted(agg.items(), key=lambda kv: kv[1]['help']-kv[1]['hurt'], reverse=True)
        print(f'{"cặp":<16}{"vi phạm":>10}{"help":>9}{"hurt":>9}{"help-hurt":>11}{"net %":>9}')
        print('-'*64)
        for (a, b), c in rank[:topn]:
            net = c['help'] - c['hurt']
            print(f'{a+arrow+b:<16}{c["viol"]:>10}{c["help"]:>9}{c["hurt"]:>9}'
                  f'{net:>11}{100.0*net/n_frames:>8.3f}%')
        print(f'... {len(agg)} cặp, {sum(1 for _,c in agg.items() if c["help"]>c["hurt"])} cặp có help > hurt')

    report(agg_excl, 'LOẠI TRỪ  (detector bật cả hai)   <-- TÍN HIỆU CHÍNH', ' - ')
    report(agg_impl, 'KÉO THEO  (i bật, j tắt)          <-- chỉ có nghĩa SAU khi lọc lift', ' -> ')

    r_help, r_hurt = fr_help/n_frames, fr_hurt/n_frames
    r_net = (fr_help-fr_hurt)/n_frames
    print(f'\n{"="*64}')
    print(f'Đã duyệt {n_runs} run · tổng {n_frames} frame')
    print(f'Frame có >=1 luật SỬA được : {fr_help:>8}  ({100*r_help:.3f}%)')
    print(f'Frame có >=1 luật PHÁ hỏng : {fr_hurt:>8}  ({100*r_hurt:.3f}%)')
    print(f'Dư địa ròng (phân biệt)    : {fr_help-fr_hurt:>8}  ({100*r_net:.3f}%)')

    if r_net < 0.01:
        print('\n>>> NO-GO cho kỳ vọng lớn: dư địa dưới 1% frame.')
        print('    Vẫn chạy decoder (vài phút) nhưng đừng kỳ vọng quá +0.3 điểm macro-F1.')
        print('    Chuyển trọng tâm đóng góp sang chế độ ít nhãn.')
    elif r_net < 0.03:
        print('\n>>> BIÊN GIỚI: dư địa 1-3%. Chạy decoder, kỳ vọng +0.2 đến +0.5 điểm.')
    else:
        print('\n>>> CÓ CHỖ CHO LUẬT. Dựng bảng luật và chạy decoder.')

    if per_subject_help:
        print('\n--- help theo subject (xem có bị một subject lái không) ---')
        tot = sum(per_subject_help.values())
        for sb, c in sorted(per_subject_help.items(), key=lambda kv: -kv[1])[:8]:
            print(f'  {sb:<12} {c:>8}  ({100.0*c/tot:.1f}%)')
        top = max(per_subject_help.values())/tot
        if top > 0.35:
            print(f'\n  CẢNH BÁO: một subject chiếm {100*top:.0f}% tổng help — kết luận không ổn định.')

## 6. Dò delta Swin-B

Bước giảm rủi ro quan trọng nhất của kế hoạch 6 tuần: toàn bộ kế hoạch đặt cược vào giả định Swin-B cho khoảng **+1.5 điểm** trên DISFA. Paper tự đo 63.0 → 64.5 trên DISFA, nhưng trên BP4D lại **giảm** 65.3 → 64.8 — nên đây là hiệu ứng riêng của tập dữ liệu, không phải quy luật chung.

Chạy đúng cấu hình đã có số ResNet-50 để so trực tiếp: `baseline / fold2 / seed0` hiện đạt **65.03**.

In [ ]:
run_logged([sys.executable, TRAIN_PY, 'train', '--help'])

In [ ]:
PROBE = dict(variant='baseline', fold=2, seed=0, arc='swin_transformer_base')
out = RUNS.parent / 'runs_swin_probe' / PROBE['variant'] / f'fold{PROBE["fold"]}' / f'seed{PROBE["seed"]}'
out.mkdir(parents=True, exist_ok=True)

print('Mốc so sánh: ResNet-50 baseline/fold2/seed0 = 65.03 test macro-F1')
print(f'Kết quả Swin-B sẽ ở: {out}\n')
cmd = [sys.executable, TRAIN_PY, 'train',
       '--arc', PROBE['arc'],
       '--variant', PROBE['variant'],
       '--fold', PROBE['fold'], '--seed', PROBE['seed'],
       '--data-root', DATA, '--manifest', SPLITS/f'fold{PROBE["fold"]}.json',
       '--output', out, '--batch-size', 32]
print('$', ' '.join(str(c) for c in cmd))
print('\n# Bỏ comment dòng dưới để chạy — khoảng 18 giờ')
print('# run_logged(cmd)')
print('\nĐỐI CHIẾU tên tham số với output ô --help phía trên trước khi chạy.')
print('batch 32 thay vì 64 vì Swin-B nặng hơn, dễ hết VRAM.')

## Checklist tuần 1

- [ ] Ô 0 in `>>> Đường dẫn hợp lệ.`
- [ ] Bước 2: `evaluation/ KHÔNG đổi — an toàn`
- [ ] Bước 3: số run có val **bằng** số run có `best.pt` (hiện là 9)
- [ ] Bước 4: `KHÔNG CÓ VẤN ĐỀ`
- [ ] Bước 5: có con số dư địa ròng → quyết mức đầu tư cho Tầng 2
- [ ] Bước 6: đối chiếu `train --help`, rồi khởi động run Swin-B
- [ ] Chẩn đoán `best_epoch ≤ 2` *(chưa có ở đây — cần xem `history.json` trước)*

**Chưa khởi động run seed 1 nào** cho tới khi chẩn đoán xong `best_epoch`. Nếu kết luận là phải đổi cách chọn checkpoint hoặc phải refit trên `fit + inner val`, thì 9 run seed 0 hiện có thành vô giá trị và phải chạy lại.

Gửi lại output bước 4 và bước 5, kèm một file `history.json`.